# Реплікація таблиці 8 Da (2025)

P05: вісім зафіксованих OLS-моделей, окремо mat/por. [Протокол](../docs/replication_protocol.md), [K1](../docs/K1_readiness.md). Основний inference — HC3/use_t=false; SE автора невідомий. Дані, фільтри й категорії P03 незмінні. Restart Kernel and Run All Cells.

## 1. Актуальність входів і повний запуск

Workflow спочатку записує running marker, перевіряє середовище, P03/P04 та еталони. Контрольна M1 проходить перевірку lstsq перед усіма моделями. За будь-якої помилки статус failed; старі результати не є актуальними. Новий Git HEAD за незмінних content hashes дозволений.

In [ ]:
from IPython.display import Image, Markdown, display

from parentedu.paths import ROOT
from parentedu.replication_workflow import run_replication

run = run_replication()
print(f"Стан реплікації: {run['metadata']['status']}")

## 2. Моделі та коефіцієнти

Константа, Medu/Fedu як категорії з reference=1. Контролі додаються послідовно: age/sex; school; guardian. Не додаємо G1/G2, studytime, absences або failures. Один набір рядків на предмет; G3=0 збережено. Еталону intercept немає, але наша оцінка збережена.

In [ ]:
display(run["tables"]["models"])
coefficients = run["tables"]["coefficients"]
display(coefficients.loc[coefficients["covariance_type"].eq("HC3")])

## 3. Числовий збіг та невідомі авторські SE

Estimate/N/adjusted R² зіставляються окремо від SE, з допусками точності друку. SE-таблиця містить усі наперед обрані методи HC3/nonrobust/HC1. Найближчий метод не оголошуємо встановленим авторським методом.

In [ ]:
display(run["tables"]["verdicts"])
comparison = run["tables"]["comparison"]
display(comparison.loc[~comparison["matches"]])
display(run["tables"]["se_comparison"].groupby("covariance_type")["matches"].agg(["sum", "count"]))

## 4. H1/H2 і діагностика

H1 та H2 мають по два двобічні тести з Holm окремо всередині сімейства. H2 використовує повну коваріацію для прямого контрасту. CI — звичайні 95% normal, не одночасні. Прапорці VIF/Cook/leverage не видаляють рядки.

In [ ]:
display(run["tables"]["hypotheses"])
display(run["tables"]["diagnostics"])
display(run["tables"]["vif"])
for name, path in run["figures"].items():
    if name.endswith("_png"):
        display(Image(filename=str(path)))

## 5. Три типи висновків

Текст нижче походить із CSV: числова реплікація, авторський SE/inference та наш змістовий висновок. Це асоціації, без причинного ефекту. R1–R5, H3 зі studytime та плацебо залишаються для P06.

In [ ]:
display(Markdown((ROOT / "docs/replication_findings.md").read_text(encoding="utf-8")))
print("Метадані: outputs/logs/replication_run.json")

Після незалежного аудиту рішення про завершення реплікації записується у [K2](../docs/K2_replication.md). Відкрита бібліографічна вимога курсу й невідомий тип авторських SE лишаються явними.